# Service Analytics - Data Exploration

This notebook explores the service metrics data, performs data profiling, and identifies key patterns.

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully')

## 1. Connect to PostgreSQL Database

In [ ]:
# Database connection
conn = psycopg2.connect(
    host='postgres',
    database='analytics',
    user='admin',
    password='admin123'
)

# Load data
query = '''
SELECT * FROM daily_metrics
WHERE metric_date >= CURRENT_DATE - INTERVAL '90 days'
ORDER BY metric_date DESC
'''

df_metrics = pd.read_sql(query, conn)
print(f'Loaded {len(df_metrics)} records')
print(f'Date range: {df_metrics["metric_date"].min()} to {df_metrics["metric_date"].max()}')

## 2. Data Profiling

In [ ]:
# Data shape and info
print(f'Shape: {df_metrics.shape}')
print(f'\nData Types:')
print(df_metrics.dtypes)
print(f'\nMissing Values:')
print(df_metrics.isnull().sum())
print(f'\nBasic Statistics:')
df_metrics.describe()

## 3. Key Metrics Analysis

In [ ]:
# Calculate success rate
df_metrics['success_rate'] = (df_metrics['completed_requests'] / df_metrics['total_requests'] * 100).round(2)

# Summary statistics
print('Total Requests:', df_metrics['total_requests'].sum())
print('Completed Requests:', df_metrics['completed_requests'].sum())
print('Failed Requests:', df_metrics['failed_requests'].sum())
print('Average Success Rate:', df_metrics['success_rate'].mean().round(2), '%')
print('Average Resolution Time:', df_metrics['avg_resolution_time_minutes'].mean().round(2), 'minutes')
print('Average Satisfaction Score:', df_metrics['avg_customer_satisfaction'].mean().round(2))
print('Total Revenue:', f"${df_metrics['revenue'].sum():.2f}")

## 4. Service Comparison

In [ ]:
# Group by service
service_stats = df_metrics.groupby('service_id').agg({
    'total_requests': 'sum',
    'completed_requests': 'sum',
    'failed_requests': 'sum',
    'avg_resolution_time_minutes': 'mean',
    'avg_customer_satisfaction': 'mean',
    'revenue': 'sum'
}).round(2)

service_stats['success_rate'] = (service_stats['completed_requests'] / service_stats['total_requests'] * 100).round(2)
print(service_stats)

# Top performing service
top_service = service_stats['revenue'].idxmax()
print(f'\nTop Revenue Service: {top_service} (${service_stats.loc[top_service, "revenue"]:.2f})')

## 5. Visualizations

In [ ]:
# Distribution of total requests
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Requests distribution
axes[0, 0].hist(df_metrics['total_requests'], bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Total Requests', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Total Requests')
axes[0, 0].set_ylabel('Frequency')

# Satisfaction distribution
axes[0, 1].hist(df_metrics['avg_customer_satisfaction'], bins=20, color='green', edgecolor='black')
axes[0, 1].set_title('Distribution of Customer Satisfaction', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Satisfaction Score (1-5)')
axes[0, 1].set_ylabel('Frequency')

# Resolution time distribution
axes[1, 0].hist(df_metrics['avg_resolution_time_minutes'], bins=30, color='orange', edgecolor='black')
axes[1, 0].set_title('Distribution of Resolution Time', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Minutes')
axes[1, 0].set_ylabel('Frequency')

# Revenue distribution by service
service_revenue = df_metrics.groupby('service_id')['revenue'].sum().sort_values(ascending=True)
axes[1, 1].barh(service_revenue.index, service_revenue.values, color='purple', edgecolor='black')
axes[1, 1].set_title('Total Revenue by Service', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Revenue ($)')

plt.tight_layout()
plt.show()

print('Visualizations generated successfully')

## 6. Data Quality Checks

In [ ]:
# Outlier detection
Q1 = df_metrics['total_requests'].quantile(0.25)
Q3 = df_metrics['total_requests'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_metrics[(df_metrics['total_requests'] < Q1 - 1.5*IQR) | (df_metrics['total_requests'] > Q3 + 1.5*IQR)]

print(f'Outliers detected: {len(outliers)}')
print(f'Success rate consistency: {(df_metrics["success_rate"] >= 0) & (df_metrics["success_rate"] <= 100).all()}')
print(f'Negative values found: {(df_metrics.select_dtypes(include=[np.number]) < 0).any().any()}')
print(f'Completed requests <= Total requests: {(df_metrics["completed_requests"] <= df_metrics["total_requests"]).all()}')

## 7. Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = ['total_requests', 'completed_requests', 'failed_requests', 
                'avg_resolution_time_minutes', 'avg_customer_satisfaction', 'revenue']
correlation = df_metrics[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Key Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation Analysis Complete')